# Test Features/Functions
This is a temp notebook to test out functions and features required for the DECONVersation package. Using a step by step guide on how a user would use this package. <br>
<b> Note </b>: This might be converted to a tutorial upon package deployment

In [1]:
#import warnings
#warnings.filterwarnings('ignore')

# Step 1: Prep Data

In [2]:
# For reading and processing pseudobulk
from pseudobulk import *

# For pre-processing h5ad for geneformer or cell2sentence
from preprocessing import *

In [3]:
# Path to pseudobulk
# Contains two samples that are essentially duplicates (pre-menopausal)
path_1 = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/geneformer/complete/haoSub.h5ad"

### For Geneformer

In [4]:
import scanpy as sc
# Read the h5ad file (just to explore columns and variables)
adata_1 = sc.read_h5ad(path_1)

adata_1 = adata_1[adata_1.obs["broad_type"].isin(
    ['B cells', 'Monocytes', 'NK cells', 'T cells', 'mDC', 'pDC']
)]

# Prep data for geneformer (first data set)
adata_1_gf = adata_1.copy()

# Remove unmapped genes
adata_1_gf = adata_1_gf[:, adata_1_gf.var.index.notnull()]

# Prep data
adata_1_gf = load_and_prep_data(adata= adata_1_gf, cell_type_col= "broad_type", mode="geneformer")

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv/src/DECONVersation/preprocessing.py:50: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cell_id"] = adata.obs.index


### For Cell2Sentence

In [5]:
# Prep data for geneformer (first data set)
adata_1_c2s = adata_1.copy()

# Limit to only select cell types
adata_1_c2s.var.index = gene_id_name_map(gene_list=adata_1_c2s.var.index,
                                      mode="to_symbol" )

# Remove unmapped genes
adata_1_c2s = adata_1_c2s[:, adata_1_c2s.var.index.notnull()]

# Prep data for Cell2Sentence
adata_1_c2s = load_and_prep_data(adata = adata_1_c2s,
                                 cell_type_col = "broad_type", 
                                 mode = "c2s",
                                 organism = "Homo Sapiens")

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv/src/DECONVersation/preprocessing.py:82: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["organism"] = organism


### Load Bulk Data

In [6]:
hoek_bulk_gf = pd.read_csv("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/hoek/hoek_counts_id_transpose.csv",
                       index_col=0)

hoek_bulk_c2s = pd.read_csv("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/hoek/hoek_counts.csv",
                       index_col=0)

In [7]:
hoek_bulk_gf = hoek_bulk_c2s.copy()
hoek_bulk_gf.index = gene_id_name_map(gene_list=hoek_bulk_gf.index, mode="to_ensembl" )

In [8]:
hoek_bulk_gf = hoek_bulk_gf.loc[hoek_bulk_gf.index.dropna()]

# Step 2: Create Signature Matrix

In [9]:
adata_1_gf

AnnData object with n_obs × n_vars = 144791 × 22982
    obs: 'cell_ID', 'cell_ID.old', 'cell_type', 'nReads_SimBu', 'nGenes_SimBu', 'batch', 'orig.ident', 'broad_type', 'cell_id', 'cell_index', 'type', 'n_counts'
    var: 'gene_name', 'gene_id', 'gene_id_long', 'ensembl_id'

In [10]:
sig_mat_gf = create_signature_matrix(adata = adata_1_gf,
                                     sample_col = "batch", # sample id column
                                     celltype_col = "broad_type",
                                     groupby = "broad_type",
                                     output_path = None)

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv/src/DECONVersation/preprocessing.py:239: UserWarning: Expression matrix contains non-integer values. Raw counts are expected. 
  warnings.warn(


In [ ]:
sig_mat_c2s = create_signature_matrix(adata = adata_1_c2s,
                                     sample_col = "batch", # sample id column
                                     celltype_col = "broad_type",
                                     groupby = "broad_type",
                                     output_path = None)

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv/src/DECONVersation/preprocessing.py:239: UserWarning: Expression matrix contains non-integer values. Raw counts are expected. 
  warnings.warn(


# Step 3: Extract Geneformer Embeddings

In [10]:
from embeddings import *

geneformer is not installed. Skipping related functions.


/nfs/sw/easybuild/software/custom-conda/cell2sentence-1.0/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cell2sentence successfully imported.


In [34]:
sig_mat_gf_embed = extract_embs(
    bulk_df = sig_mat_gf.T,
    mode = "geneformer",
    temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
    model_path= "ctheodoris/Geneformer",
    delete_temp_files = True
)

Pseudobulk AnnData saved to: /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad
Starting Geneformer tokenization...
Tokenizing /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad


100%|██████████| 1/1 [00:00<00:00, 21.25it/s]

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.



/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:740: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['B cells', 'Monocytes', 'NK cells', 'T cells', 'mDC']

    Inferred to be: categorical

  value_idx = self._prep_dim_index(value.index, attr)
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:787: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  getattr(self, attr).index = value
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/sw/easybuild/software/custom-conda/geneformer-1.0/lib/python3.10/site-packages/geneformer/tokenizer.py:473: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated a

Loading Geneformer model...


/nfs/sw/easybuild/software/custom-conda/geneformer-1.0/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(
CLS token present in token dictionary, excluding from average.
EOS token present in token dictionary, excluding from average.


Loading tokenized dataset...
Extracting Geneformer embeddings...


  0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
gf_embed = extract_embs(
    bulk_df = hoek_bulk_gf.T,
    mode = "geneformer",
    temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
    model_path= "ctheodoris/Geneformer",
    delete_temp_files = True
)

Pseudobulk AnnData saved to: /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/gf_tokens.h5ad
Starting Geneformer tokenization...
Tokenizing /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/gf_tokens.h5ad


100%|██████████| 1/1 [00:00<00:00, 153.78it/s]

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/gf_tokens.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.



/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:787: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  getattr(self, attr).index = value
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/nfs/sw/easybuild/software/custom-conda/geneformer-1.0/lib/python3.10/site-packages/geneformer/tokenizer.py:473: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/nfs/sw/easybuild/software/custom-conda/geneformer-1.0/lib/python3.10/site-packages/geneformer/tokenizer.py:476: FutureWarning: Series.__getitem__ tr

Loading Geneformer model...


/nfs/sw/easybuild/software/custom-conda/geneformer-1.0/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(
CLS token present in token dictionary, excluding from average.
EOS token present in token dictionary, excluding from average.


Loading tokenized dataset...
Extracting Geneformer embeddings...


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#sig_mat_gf_embed.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/gf_sig_matrix_embed_hoek.csv")
#gf_embed.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/gf_bulk_embed_hoek.csv")

# Step 4: Extract Cell2Sentence Embeddings

In [11]:
from embeddings import *

In [16]:
sig_mat_c2s_embed = extract_embs(
    bulk_df = sig_mat_c2s.T,
    mode = "c2s",
    temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
    model_path= "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/2b/2026-02-02-13_12_05_finetune_cell_type_prediction/checkpoint-3800",
    delete_temp_files = True
)

#/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/hao800/2026-02-01-00_05_11_finetune_cell_type_prediction/checkpoint-3800
#/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/2b/2026-02-02-13_12_05_finetune_cell_type_prediction/checkpoint-3800

In [18]:
# c2s_embed = extract_embs(
#     bulk_df = hoek_bulk_c2s.T,
#     mode = "c2s",
#     temp_output_dir = "/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/", 
#     model_path= "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/pbmc/c2s/2b/2026-02-02-13_12_05_finetune_cell_type_prediction/checkpoint-3800",
#     delete_temp_files = True
# )

In [19]:
#sig_mat_c2s_embed.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/c2s_2b_sig_matrix_embed_hoek.csv")
#c2s_embed.to_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/c2s_2b_bulk_embed_hoek.csv")

<b> Features to add or fix </b> <br> 
Add chunk size or batch size for both c2s and geneformer <br>
Possibly split geneformer and c2s into different functions without using a wrapper <br>
For geneformer, expose user to select which embedding dimension and type (cls vs cell) to extarct <br>
For cell2sentence, expose user to how many input dimensions to match

# Step 5: Run NNLS

In [84]:
# sig_mat_gf_embed = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/gf_sig_matrix_embed_hoek.csv", index_col=0)
# gf_embed = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/gf_bulk_embed_hoek.csv", index_col=0)

sig_mat_c2s_embed = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/c2s_2b_sig_matrix_embed_hoek.csv", index_col=0)
c2s_embed = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/c2s_2b_bulk_embed_hoek.csv", index_col=0)

In [85]:
ground_truth = pd.read_csv("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/hoek/hoek_facs.csv",
                          index_col=0)

### Geneformer embeddings

In [86]:
from deconvolution import *

In [87]:
cell_prop_pred = run_deconv_nnls(
    bulk_df = c2s_embed.T,
    signature_df = sig_mat_c2s_embed.T,
    normalize = True)

Using 2304 common features.
Running NNLS deconvolution...


In [100]:
c2s_embed.head()

,C2S_0,C2S_1,C2S_2,C2S_3,C2S_4,C2S_5,C2S_6,C2S_7,C2S_8,C2S_9,...,C2S_2294,C2S_2295,C2S_2296,C2S_2297,C2S_2298,C2S_2299,C2S_2300,C2S_2301,C2S_2302,C2S_2303
name,,,,,,,,,,,,,,,,,,,,,
HD30_PBMC_0,-0.618390,0.073844,-0.184268,-0.044321,-1.141646,-0.718154,-0.582614,0.844262,-1.492359,-2.723187,...,-0.420479,0.836745,-2.126173,0.750428,1.050359,0.864871,2.149834,-2.477567,1.782817,-0.939739
HD30_PBMC_1,-0.738623,0.156094,-0.357581,-0.228437,-1.251178,-0.571320,-0.914015,0.652387,-1.880877,-2.407562,...,-0.656804,0.879802,-1.743491,0.718747,1.112239,1.350723,2.238795,-2.723862,2.017000,-1.013223
HD30_PBMC_3,-0.551069,-0.012240,0.086905,-0.517794,-1.029959,-0.494358,-0.421795,0.694847,-1.358882,-2.847907,...,-0.323960,0.942814,-2.181475,0.833559,0.904186,0.647618,1.984324,-2.141453,1.641776,-0.931973
HD30_PBMC_7,-1.166236,0.252180,0.176336,-0.407395,-1.092631,-0.911738,-0.657478,0.838774,-1.482954,-2.934255,...,-0.325897,1.068095,-2.087025,0.739559,0.949851,0.674165,2.039067,-1.882667,1.566737,-0.973005
HD31_PBMC_0,-0.604917,-0.098051,0.301460,0.021781,-0.721387,-0.613019,-0.445544,0.514093,-1.494422,-2.795332,...,-0.503513,0.725193,-1.949751,0.846829,0.609580,0.097967,1.784992,-2.172966,1.710262,-0.856949


### Cell2Sentence embeddings

In [88]:
# cell_prop_pred_c2s = run_deconv_nnls(
#     bulk_df = c2s_embed.T,
#     signature_df = sig_mat_c2s_embed.T,
#     normalize = True)

# Step 7: Evaluate 

In [89]:
from evaluation import *

### Geneformer

In [90]:
cell_prop_pred["mDC"] = cell_prop_pred["pDC"] + cell_prop_pred["mDC"]
cell_prop_pred = cell_prop_pred.drop(columns=["pDC"])
cell_prop_pred = cell_prop_pred.rename(columns={"T cells" : "T cell"})

In [91]:
rmse_per_cell_type = compute_rmse(true_df = ground_truth.T,
                                 pred_df = cell_prop_pred,
                                 return_per_celltype = True)

rmse_per_cell_type = pd.DataFrame(rmse_per_cell_type)

In [95]:
rmse_per_cell_type

,overall,per_celltype
T cell,0.162728,0.175725
Monocytes,0.162728,0.308203
B cells,0.162728,0.071004
mDC,0.162728,0.021488
NK cells,0.162728,0.032082


In [93]:
corr_per_cell_type = compute_correlation(true_df = ground_truth.T,
                                         pred_df = cell_prop_pred,
                                        return_per_celltype = True)

corr_per_cell_type = pd.DataFrame(corr_per_cell_type)

/nfs/home/aoku/.local/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/nfs/home/aoku/.local/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [94]:
corr_per_cell_type

,overall,per_celltype
T cell,0.699647,0.846658
Monocytes,0.699647,0.820932
B cells,0.699647,NaN
mDC,0.699647,0.667173
NK cells,0.699647,0.892776


In [29]:
# Cell2Sentence

# cell_prop_pred_c2s["mDC"] = cell_prop_pred_c2s["pDC"] + cell_prop_pred_c2s["mDC"]
# cell_prop_pred_c2s = cell_prop_pred_c2s.drop(columns=["pDC"])
# cell_prop_pred_c2s = cell_prop_pred_c2s.rename(columns={"T cells" : "T cell"})

# Temp - Delete later (for testing only)

In [33]:
import sys
import importlib

# 1. Clear the 'embeddings' entry from the internal cache
if 'embeddings' in sys.modules:
    del sys.modules['embeddings']

# 2. Re-import the fresh version
import embeddings

# 3. Pull the updated functions into your current namespace
from embeddings import *

geneformer successfully imported.
cell2sentence is not installed. Skipping related functions.


In [26]:
finetuned_models = pd.read_csv("/nfs/home/rfu/projects/deconv_language/finetunings.csv")

In [27]:
finetuned_models

,time,dir,model,param,dataset,ncell,method
0,225.570417,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,czi_sub,9788,C2S_2B
1,83.050862,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,410M,czi_sub,9788,C2S_410M
2,9.281267,/gpfs/commons/groups/compbio/projects/rf_proje...,gf,316M,czi_sub,9788,GF_316M
3,107.726501,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,deconvBench,4800,C2S_2B
4,70.739383,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,410M,deconvBench,4800,C2S_410M
5,15.143393,/gpfs/commons/groups/compbio/projects/rf_proje...,gf,316M,deconvBench,4800,GF_316M
6,93.318095,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,GSE220608,4140,C2S_2B
7,13.215372,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,410M,GSE220608,4140,C2S_410M
8,16.258566,/gpfs/commons/groups/compbio/projects/rf_proje...,gf,316M,GSE220608,4140,GF_316M
9,1286.171553,/gpfs/commons/groups/compbio/projects/rf_proje...,c2s,2B,LIBD,56447,C2S_2B


### Merge GF and C2S results

In [2]:
import pandas as pd

In [37]:
c2s_embed = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/c2s_2b_bulk_embed_hoek.csv",
                       index_col=0)

c2s_embed_sig_mat = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/c2s_2b_sig_matrix_embed_hoek.csv",
                       index_col=0)

In [61]:
gf_embed = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/gf_bulk_embed_hoek.csv",
                       index_col=0)

gf_embed_sig_mat = pd.read_csv("/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/hoek/gf_sig_matrix_embed_hoek.csv",
                       index_col=0)

gf_embed_sig_mat = gf_embed_sig_mat.loc[c2s_embed_sig_mat.index, ]

In [80]:
ground_truth = pd.read_csv("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/hoek/hoek_facs.csv", index_col=0)

In [81]:
ground_truth.head()

,HD30_PBMC_0,HD30_PBMC_1,HD30_PBMC_3,HD30_PBMC_7,HD31_PBMC_0,HD31_PBMC_1,HD31_PBMC_3,HD31_PBMC_7
T cell,0.5140,0.4570,0.5800,0.5780,0.5900,0.6230,0.6540,0.6360
Monocytes,0.1550,0.2140,0.1530,0.1350,0.1810,0.1700,0.1750,0.1440
B cells,0.0430,0.0480,0.0420,0.0660,0.0910,0.0930,0.0610,0.0970
mDC,0.0264,0.0251,0.0172,0.0184,0.0163,0.0152,0.0133,0.0161
NK cells,0.1780,0.1680,0.1480,0.1320,0.0450,0.0340,0.0490,0.0360


In [63]:
# Merge both
c2s_gf_merged = pd.concat([c2s_embed, gf_embed], axis=1)
c2s_gf_merged_sig_mat = pd.concat([c2s_embed_sig_mat, gf_embed_sig_mat], axis=1)

In [70]:
# Run NNLS
cell_prop_pred = run_deconv_nnls(
    bulk_df = c2s_gf_merged.T,
    signature_df = c2s_gf_merged_sig_mat.T,
    normalize = True)

Using 3456 common features.
Running NNLS deconvolution...


In [71]:
cell_prop_pred["mDC"] = cell_prop_pred["pDC"] + cell_prop_pred["mDC"]
cell_prop_pred = cell_prop_pred.drop(columns=["pDC"])
cell_prop_pred = cell_prop_pred.rename(columns={"T cells" : "T cell"})

In [72]:
cell_prop_pred.head()

name,B cells,Monocytes,NK cells,T cell,mDC
HD30_PBMC_0,0.0,0.456077,0.165365,0.348950,0.029608
HD30_PBMC_1,0.0,0.528104,0.225446,0.170773,0.075678
HD30_PBMC_3,0.0,0.415139,0.130566,0.454295,0.000000
HD30_PBMC_7,0.0,0.397429,0.082694,0.519877,0.000000
HD31_PBMC_0,0.0,0.529449,0.061128,0.388681,0.020742


In [74]:
rmse_per_cell_type_merged = compute_rmse(true_df = ground_truth.T,
                                 pred_df = cell_prop_pred,
                                 return_per_celltype = True)

rmse_per_cell_type_merged = pd.DataFrame(rmse_per_cell_type_merged)

rmse_per_cell_type_merged

,overall,per_celltype
T cell,0.164102,0.180004
Monocytes,0.164102,0.309553
B cells,0.164102,0.071004
mDC,0.164102,0.021164
NK cells,0.164102,0.030551


In [75]:
rmse_per_cell_type

,overall,per_celltype
T cell,0.175443,0.187477
Monocytes,0.175443,0.334024
B cells,0.175443,0.071004
mDC,0.175443,0.022107
NK cells,0.175443,0.040638


In [78]:
corr_per_cell_type_merged = compute_correlation(true_df = ground_truth.T,
                                         pred_df = cell_prop_pred,
                                        return_per_celltype = True)

corr_per_cell_type_merged = pd.DataFrame(corr_per_cell_type_merged)

corr_per_cell_type_merged

/nfs/home/aoku/.local/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/nfs/home/aoku/.local/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,overall,per_celltype
T cell,0.691671,0.844395
Monocytes,0.691671,0.818172
B cells,0.691671,NaN
mDC,0.691671,0.611642
NK cells,0.691671,0.895732


In [76]:
corr_per_cell_type

,overall,per_celltype
T cell,0.643422,0.952552
Monocytes,0.643422,0.704390
B cells,0.643422,NaN
mDC,0.643422,0.387216
NK cells,0.643422,0.872409


In [79]:
ground_truth

,HD30_PBMC_0,HD30_PBMC_1,HD30_PBMC_3,HD30_PBMC_7,HD31_PBMC_0,HD31_PBMC_1,HD31_PBMC_3,HD31_PBMC_7
T cell,0.5140,0.4570,0.5800,0.5780,0.5900,0.6230,0.6540,0.6360
Monocytes,0.1550,0.2140,0.1530,0.1350,0.1810,0.1700,0.1750,0.1440
B cells,0.0430,0.0480,0.0420,0.0660,0.0910,0.0930,0.0610,0.0970
mDC,0.0264,0.0251,0.0172,0.0184,0.0163,0.0152,0.0133,0.0161
NK cells,0.1780,0.1680,0.1480,0.1320,0.0450,0.0340,0.0490,0.0360
